# linear Model regression and improve it

### import  libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import os

warnings.simplefilter(action='ignore')
plt.style.use ('seaborn')

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### loading data 

In [ ]:
#loading data 
data = pd.read_csv("/kaggle/input/cardata/cardata.csv")
data

### Check data and preprocesing

In [ ]:
#change data to data frame
df= pd.DataFrame (data)

In [ ]:
# checking the number of rows and columns
df.shape

In [ ]:
#descibtion data fram 
df.describe(include='all')

In [ ]:
#number of null values in our dataset
df.isnull().sum()

In [ ]:
#getting some information about the datafram
df.info()

In [ ]:
#checking for dublicate row
df.duplicated().sum()

In [ ]:
#dropping duplicates

# I made model by droping duplicates data but the score was bad. so I do not remowe them.

In [ ]:
#drop car name
df2 =df.drop(['Car_Name'], axis=1) 
df2

In [ ]:
# find maximum of year column
a=df2['Year'].max()
a

In [ ]:
# max year+1(2019)
# so calcute the difference between year and 2019 = Age
#so drop column year 

df2['Age'] = 2019 - df2['Year'] 
df3 =df2.drop(['Year'],axis = 1)
df3

In [ ]:
#checking the distribution of categorical data
print(df3['Seller_Type'].value_counts())

In [ ]:
print(df3['Transmission'].value_counts())

In [ ]:
print(df3['Fuel_Type'].value_counts())

In [ ]:
# checking categorical data to target
from matplotlib import style

In [ ]:
# plot categorical data with target
style.use ('ggplot')
fig=plt.figure(figsize=(15,5))
fig.suptitle('visualizing categorical data columns', fontsize=20)

plt.subplot(1,3,1)
plt.bar(df3['Fuel_Type'], df3['Selling_Price'],color= 'green')
plt.xlabel('Fuel_Type')
plt.ylabel('Selling_Price')

plt.subplot(1,3,2)
plt.bar(df3['Seller_Type'], df3['Selling_Price'],color= 'red')
plt.xlabel('Seller_Type')
plt.ylabel('Selling_Price')

plt.subplot(1,3,3)
plt.bar(df3['Transmission'], df3['Selling_Price'],color= 'purple')
plt.xlabel('Transmission')
plt.ylabel('Selling_Price')
plt.show()

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,5))

fig.suptitle('visualizing categorical data columns', fontsize=20)
sns.barplot(x=df3['Fuel_Type'],y= df3['Selling_Price'], ax=axes[0])
sns.barplot(x=df3['Seller_Type'],y= df3['Selling_Price'], ax=axes[1])
sns.barplot(x=df3['Transmission'],y= df3['Selling_Price'], ax=axes[2])

In [ ]:
df3.corr()

### result of correlation in numerical data
present price>Age>owner>kms_driven

### check catrgorical feature

In [ ]:
# Cheking fueltype
df3.Fuel_Type.unique()

In [ ]:
#Manual Encoding for fueltype
#Petrol==2,Diesel==3,CNG==4
df3.replace({'Fuel_Type': {'Petrol':2,'Diesel':3,'CNG':4}} , inplace=True)

In [ ]:
# cheking Seller_Type
df3.Seller_Type.unique()

In [ ]:
#Manual Encoding for Seller_Type
#Dealer==2,Individual==3
df3.replace({'Seller_Type': {'Dealer':2,'Individual':3}} , inplace=True)

In [ ]:
# cheking Transmission
df3.Transmission.unique()

In [ ]:
#Manual Encoding for Transmission
#Manual==2,Automatic==3
df3.replace({'Transmission': {'Manual':2,'Automatic':3}} , inplace=True)

In [ ]:
df3.describe()

In [ ]:
df3.info()

### Checking for noise with scatter plot

In [ ]:
#Checking for noise with scatter plot
plt.figure(figsize=(20,8))

plt.subplot(2,2,1)
sns.histplot(df3.Present_Price)
plt.xlabel( 'Present_Price', fontsize=16)
plt.ylabel( 'Selling_Price', fontsize=16)

plt.subplot(2,2,2)
sns.histplot(df3.Kms_Driven)
plt.xlabel( 'Kms_Driven', fontsize=16)
plt.ylabel( 'Selling_Price', fontsize=16)



plt.subplot(2,2,3)
plt.scatter(df3.Present_Price, df3.Selling_Price)
plt.xlabel( 'Present_Price', fontsize=16)
plt.ylabel( 'Selling_Price', fontsize=16)

plt.subplot(2,2,4)
plt.scatter(df3.Kms_Driven, df3.Selling_Price)
plt.xlabel( 'Kms_Driven', fontsize=16)
plt.ylabel( 'Selling_Price', fontsize=16)


In [ ]:
# scatter plot for age and target
plt.figure(figsize=(20,8))

plt.subplot(1,2,1)
sns.histplot(df3.Age)
plt.xlabel( 'Age', fontsize=16)
plt.ylabel( 'Selling_Price', fontsize=16)

plt.subplot(1,2,2)
plt.scatter(df3.Age, df3.Selling_Price)
plt.xlabel( 'Age', fontsize=16)

plt.ylabel( 'Selling_Price', fontsize=16)

In [ ]:
plt.figure(figsize=(20,8))

plt.subplot(1,2,1)
plt.title('Car Price distribution')
sns.histplot(df3.Selling_Price)

plt.subplot(1,2,2)
plt.title('Car Price Spread')
sns.boxplot(y = df3.Selling_Price, orient="v", color='orange')

#### # I think there Are True outlier in selling price and Kms_driven. they are logical data and i can not remove them.


In [ ]:
#correlation data
plt.figure(figsize= (10,7))
sns.heatmap(df3.corr(),cmap='coolwarm', annot=True)
plt.title('correlation between the columns')
plt.tight_layout()
plt.show()

### result of correlation
#present price>seller type >fuel type>transmission>age>owner>kms_ driven 

## create model

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

In [ ]:
# spilitting training and test data
#80 -20 #train =80, test=20
x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.2, random_state=0)

In [ ]:
model=LinearRegression()
model.fit(x_train,y_train)

In [ ]:
y_pred= model.predict(x_test)
y_pred

In [ ]:
print (model.intercept_)
print (model.coef_)

In [ ]:
compare= pd.DataFrame({'Actual': y_test, 'predict': y_pred})
compare

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
sns.regplot(x=y_pred, y=y_test)
plt.xlabel ('Predicted Price')
plt.ylabel ('Actual Price')
plt.title('Actual VS predicted price')
plt.show()

### increase dimension use result of correlation
# result of correlation
#present price>seller type >fuel type>transmission>age>owner>kms_ driven 

In [ ]:
Present_Price2= df3.Present_Price**2
df3.insert(2,'Present_Price2',Present_Price2)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

In [ ]:
# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)

In [ ]:
model4=LinearRegression()
model4.fit(x_train,y_train)

In [ ]:
y_pred= model4.predict(x_test)

In [ ]:
print (model4.intercept_)
print (model4.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
Seller_Type2= df3.Seller_Type**2
df3.insert(6,'Seller_Type2',Seller_Type2)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

In [ ]:
# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)

In [ ]:
model5=LinearRegression()
model5.fit(x_train,y_train)

In [ ]:
y_pred= model5.predict(x_test)

In [ ]:
print (model5.intercept_)
print (model5.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
#### seller_typr**2 not improve r2 score 
### drop it

In [ ]:
df3.drop(columns='Seller_Type2', inplace=True)
df3

In [ ]:
Fuel_Type2= df3.Fuel_Type**2
df3.insert(5,'Fuel_Type2',Fuel_Type2)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

In [ ]:
# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)

In [ ]:
model6=LinearRegression()
model6.fit(x_train,y_train)

In [ ]:
y_pred= model6.predict(x_test)

In [ ]:
print (model6.intercept_)
print (model6.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
Transmission2= df3.Transmission**2
df3.insert(8,'Transmission2',Transmission2)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

In [ ]:
# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)

In [ ]:
model7=LinearRegression()
model7.fit(x_train,y_train)

In [ ]:
y_pred= model7.predict(x_test)

In [ ]:
print (model7.intercept_)
print (model7.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
#### Transmission**2 not improve r2 score 
### drop it

In [ ]:
df3.drop(columns='Transmission2', inplace=True)
df3

In [ ]:
Age2= df3.Age**2
df3.insert(10,'Age2',Age2)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

In [ ]:
# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)

In [ ]:
model8=LinearRegression()
model8.fit(x_train,y_train)

In [ ]:
y_pred= model8.predict(x_test)

In [ ]:
print (model8.intercept_)
print (model8.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
#### Age**2 not improve r2 score 
### drop it

In [ ]:
df3.drop(columns='Age2', inplace=True)
df3

In [ ]:
Owner2= df3.Owner**2
df3.insert(9,'Owner2',Owner2)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

In [ ]:
# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)

In [ ]:
model9=LinearRegression()
model9.fit(x_train,y_train)

In [ ]:
y_pred= model9.predict(x_test)

In [ ]:
print (model9.intercept_)
print (model9.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
Kms_Driven2= df3.Kms_Driven**2
df3.insert(4,'Kms_Driven2',Kms_Driven2)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

In [ ]:
# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)

In [ ]:
model10=LinearRegression()
model10.fit(x_train,y_train)

In [ ]:
y_pred= model10.predict(x_test)

In [ ]:
print (model10.intercept_)
print (model10.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

### model improve +2%

### use dot product feather 
#present price & kms_driven& fuel_type&owner is important so i use them for dot product

In [ ]:
Present_Kms= df3['Present_Price']*df3['Kms_Driven']
df3.insert(3,'Present_Kms',Present_Kms)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)
model11=LinearRegression()
model11.fit(x_train,y_train)

In [ ]:
y_pred= model11.predict(x_test)
print (model11.intercept_)
print (model11.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
####R2 is good

In [ ]:
Present_Fuel= df3['Present_Price']*df3['Fuel_Type']
df3.insert(4,'Present_Fuel',Present_Fuel)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)
model12=LinearRegression()
model12.fit(x_train,y_train)

In [ ]:
y_pred= model12.predict(x_test)
print (model12.intercept_)
print (model12.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

In [ ]:
#decrease R2 ### drop Present_Fuel

In [ ]:
df3.drop(columns='Present_Fuel', inplace=True)
df3

In [ ]:
Present_kms2= df3['Present_Price']*df3['Kms_Driven2']
df3.insert(4,'Present_kms2',Present_kms2)
df3

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)
model13=LinearRegression()
model13.fit(x_train,y_train)

In [ ]:
y_pred= model13.predict(x_test)
print (model13.intercept_)
print (model13.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

### R2 increse it is good

### use cross validation to improve model

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

In [ ]:
kfold_validation=KFold (10)
result= cross_val_score (model13, x_train, y_train , cv=kfold_validation)
print (result)

## Slide 7 is bad drop it

In [ ]:
301/11

In [ ]:
df3.drop(df3.index[164:193], inplace=True)
df3= df3.reset_index(drop=True)
df3.shape

### model after cross validation

In [ ]:
#spilitting data and target
x= df3.drop (['Selling_Price'], axis=1)
y=df3['Selling_Price']

# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)
model15=LinearRegression()
model15.fit(x_train,y_train)

In [ ]:
y_pred= model15.predict(x_test)
print (model15.intercept_)
print (pd.DataFrame(model15.coef_, x.columns, columns=['coef']))

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

### R2 score is very good

In [ ]:
compare3= pd.DataFrame({'Actual': y_test, 'predict': y_pred})
compare3

In [ ]:
plt.figure(figsize= (15,10))
sns.regplot(x=y_pred, y=y_test)
plt.xlabel ('Predicted Price')
plt.ylabel ('Actual Price')
plt.title('Actual VS predicted price')
plt.show()

In [ ]:
plt.figure(figsize= (15,10))
a= x_test.Present_Price
b= y_test
c= x_test.Present_Price
d= y_pred
plt.scatter (a,b)
plt.scatter (c,d)
plt.xlabel ('Present_Price')
plt.ylabel ('Selling_Price')
plt.grid()
plt.show()

In [ ]:
#add y_test
x_test.insert (0, 'y_test', y_test)
#add y_pred
x_test.insert (0, 'y_pred', y_pred)

In [ ]:
df5= x_test.sort_values(by=['Present_Price'])
df5

In [ ]:
plt.figure(figsize= (15,10))
a= df5.Present_Price
b= df5.y_test
c=df5.Present_Price
d= df5.y_pred
plt.scatter (a,b)
plt.plot (c,d, color='blue')
plt.xlabel ('Present_Price')
plt.ylabel ('Selling_Price')

plt.show()

### Regression model with normalize

In [ ]:
###normalize
from sklearn import preprocessing

In [ ]:
###normalize
scaler=preprocessing.MinMaxScaler(feature_range=(0,1)) 
normal= scaler.fit_transform(df3)
df6= pd.DataFrame (normal, columns=[df3])
df6.head()

In [ ]:
#spilitting data and target
x= df6.drop (['Selling_Price'], axis=1)
y=df6['Selling_Price']

In [ ]:
#80 -20 #train =80, test=20
# spilitting training and test data

x_train, x_test, y_train,y_test= train_test_split(x,y, test_size= 0.20, random_state=0)
model16=LinearRegression()
model16.fit(x_train,y_train)

In [ ]:
y_pred= model16.predict(x_test)
print (model16.intercept_)
print (model16.coef_)

In [ ]:
#Error evaluation
print ('Mean Absolute Error:' , metrics.mean_absolute_error (y_test, y_pred))
print ('Mean Squard Error:' , metrics.mean_squared_error (y_test, y_pred))
print ('Root Mean Squard Error:' , np.sqrt (metrics.mean_squared_error (y_test, y_pred)))
print ('R2 Score:' ,metrics.r2_score(y_test, y_pred))

### predict sample

In [ ]:
####predict sample
#present price=11.23
#kms_driven=42000
#Fuel_type=petrol
#Sealer_type=Dealer
#Transmission= Manual
#Owner=1
#Age=10
#price=????

In [ ]:
df3.columns.tolist()

In [ ]:
Selling_Price=11
Present_Price=11.23
Present_Price2=11.23**2
Present_Kms= 11.23*42000
Present_kms2=11.23*(42000**2)
Kms_Driven=42000
Kms_Driven2=42000**2
Fuel_Type=2   # petrol code is 2
Fuel_Type2=2**2
Seller_Type=2   #Dealer code is 2
Transmission= 2  #Manual code is 2
Owner=1
Owner2=1**2
Age= 10
sample_pred= pd.DataFrame ({'Selling_Price':[Selling_Price],'Present_Price':[Present_Price],
                           'Present_Price2':[Present_Price2],'Present_Kms':[Present_Kms],
                            'Present_kms2':[Present_kms2],
                          'Kms_Driven':[Kms_Driven],'Kms_Driven2':[Kms_Driven2],'Fuel_Type':[Fuel_Type],
                          'Fuel_Type2':[Fuel_Type2],'Seller_Type':[Seller_Type],
                          'Transmission':[Transmission],
                          'Owner':[Owner], 'Owner2':[Owner2], 'Age':[Age]})
sample_pred

In [ ]:
# add sample to data frame
df_=df3.append(sample_pred)
df_

In [ ]:
#define train and test
train=df_.iloc[:272]
test= df_.iloc[272:]

In [ ]:
x_train= df_.drop (['Selling_Price'], axis=1)[:272]
y_train=df_['Selling_Price'][:272]
x_test=  df_.drop (['Selling_Price'], axis=1)[272:]
model.fit(x_train,y_train)

In [ ]:
y_pred=model.predict(x_test)
y_pred